# 🏆 Sistem Seleksi Olimpiade Matematika
## Data Mining — Klasifikasi dengan Random Forest
**Prodi: Pendidikan Matematika**

---
Notebook ini digunakan untuk:
1. Load dan eksplorasi dataset
2. Preprocessing & labeling otomatis
3. Training model Random Forest Classifier
4. Evaluasi model
5. Export model (`model.pkl` & `scaler.pkl`) untuk Streamlit


## 📦 1. Install & Import Library

In [ ]:
# Install library yang diperlukan
!pip install scikit-learn pandas openpyxl xlrd -q

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

print('✅ Semua library berhasil diimport!')

## 📂 2. Upload & Load Dataset

In [ ]:
# Upload file dataset.xlsx dari komputer kamu
from google.colab import files
uploaded = files.upload()  # Pilih file dataset.xlsx

In [ ]:
# Load dataset
df = pd.read_excel('dataset.xlsx')

# Rename kolom agar lebih mudah dibaca
df.columns = ['aljabar', 'geometri', 'bilangan', 'data_ketidakpastian', 'menalar', 'literasi']

print(f'📊 Shape dataset: {df.shape}')
print(f'\n📋 5 baris pertama:')
df.head()

## 🔍 3. Eksplorasi Data (EDA)

In [ ]:
print('=== INFORMASI DATASET ===')
print(f'Jumlah baris  : {len(df):,}')
print(f'Jumlah kolom  : {len(df.columns)}')
print(f'\nNilai kosong  :')
print(df.isnull().sum())

print(f'\n=== STATISTIK DESKRIPTIF ===')
df.describe().round(2)

In [ ]:
# Visualisasi distribusi skor
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribusi Skor Kemampuan Matematika Siswa', fontsize=14, fontweight='bold')

feature_labels = [
    ('aljabar', 'Numerasi Aljabar', '#FF6B6B'),
    ('geometri', 'Numerasi Geometri', '#4ECDC4'),
    ('bilangan', 'Numerasi Bilangan', '#45B7D1'),
    ('data_ketidakpastian', 'Data & Ketidakpastian', '#96CEB4'),
    ('menalar', 'Menalar', '#FFEAA7'),
    ('literasi', 'Literasi', '#DDA0DD'),
]

for ax, (col, label, color) in zip(axes.flat, feature_labels):
    df[col].dropna().plot(kind='hist', ax=ax, bins=30, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Skor (0-100)')
    ax.set_ylabel('Frekuensi')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('distribusi_skor.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik distribusi berhasil dibuat!')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Korelasi Antar Kemampuan Matematika', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏷️ 4. Pelabelan Data (Target Variable)

**Kriteria Labeling:**
| Label | Status | Kriteria |
|-------|--------|----------|
| 2 | ✅ Siap Olimpiade | Rata-rata ≥ 80 **DAN** Nilai minimum ≥ 65 |
| 1 | ⚡ Potensial | Rata-rata ≥ 60 **DAN** Nilai minimum ≥ 45 |
| 0 | ❌ Tidak Siap | Di luar kriteria di atas |

In [ ]:
# Bersihkan data
df_clean = df.dropna().copy()
print(f'Data setelah cleaning: {len(df_clean):,} baris (dari {len(df):,})')

FEATURES = ['aljabar', 'geometri', 'bilangan', 'data_ketidakpastian', 'menalar', 'literasi']

# Fungsi pelabelan
def label_siswa(row):
    avg = row[FEATURES].mean()
    min_score = row[FEATURES].min()
    if avg >= 80 and min_score >= 65:
        return 2  # Siap Olimpiade
    elif avg >= 60 and min_score >= 45:
        return 1  # Potensial
    else:
        return 0  # Tidak Siap

df_clean['label'] = df_clean.apply(label_siswa, axis=1)

# Distribusi label
label_counts = df_clean['label'].value_counts().sort_index()
label_names = {0: 'Tidak Siap', 1: 'Potensial', 2: 'Siap Olimpiade'}
print('\n📊 Distribusi Label:')
for k, v in label_counts.items():
    pct = v / len(df_clean) * 100
    print(f'  {label_names[k]:20s}: {v:6,} siswa ({pct:.1f}%)')

In [ ]:
# Visualisasi distribusi label
colors = ['#FF4B4B', '#FFD200', '#38EF7D']
labels_display = ['Tidak Siap', 'Potensial', 'Siap Olimpiade']
sizes = [label_counts.get(i, 0) for i in range(3)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
wedges, texts, autotexts = ax1.pie(sizes, labels=labels_display, colors=colors,
                                    autopct='%1.1f%%', startangle=90,
                                    explode=(0.02, 0.02, 0.05))
for text in autotexts:
    text.set_fontweight('bold')
ax1.set_title('Distribusi Kesiapan Olimpiade', fontweight='bold', fontsize=12)

# Bar chart
bars = ax2.bar(labels_display, sizes, color=colors, edgecolor='white', linewidth=1.5)
for bar, size in zip(bars, sizes):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             f'{size:,}', ha='center', va='bottom', fontweight='bold')
ax2.set_title('Jumlah Siswa per Kategori', fontweight='bold', fontsize=12)
ax2.set_ylabel('Jumlah Siswa')

plt.tight_layout()
plt.savefig('distribusi_label.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 5. Training Model Random Forest

In [ ]:
# Persiapan data
X = df_clean[FEATURES]
y = df_clean['label']

# Normalisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Data training : {len(X_train):,} baris')
print(f'Data testing  : {len(X_test):,} baris')

In [ ]:
# Training Random Forest
print('⏳ Training model Random Forest...')

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

print('✅ Training selesai!')

## 📈 6. Evaluasi Model

In [ ]:
# Prediksi dan evaluasi
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'🎯 AKURASI MODEL: {acc*100:.2f}%')
print('\n📋 CLASSIFICATION REPORT:')
print(classification_report(y_test, y_pred,
                             target_names=['Tidak Siap', 'Potensial', 'Siap Olimpiade']))

In [ ]:
# Confusion Matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Tidak Siap', 'Potensial', 'Siap Olimpiade'])
disp.plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title('Confusion Matrix', fontweight='bold', fontsize=12)
ax1.tick_params(axis='x', rotation=15)

# Feature importance
importance = model.feature_importances_
feat_labels = ['Aljabar', 'Geometri', 'Bilangan', 'Data & KT', 'Menalar', 'Literasi']
sorted_idx = np.argsort(importance)
colors_feat = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
ax2.barh([feat_labels[i] for i in sorted_idx],
         importance[sorted_idx],
         color=[colors_feat[i] for i in sorted_idx])
ax2.set_xlabel('Tingkat Kepentingan Fitur')
ax2.set_title('Feature Importance', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-validation score
print('⏳ Menjalankan 5-Fold Cross Validation...')
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f'CV Scores    : {[f"{s:.4f}" for s in cv_scores]}')
print(f'Mean CV      : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Akurasi Test : {acc:.4f}')

## 💾 7. Export Model

In [ ]:
# Simpan model dan scaler
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ model.pkl dan scaler.pkl berhasil disimpan!')
print('\n📌 LANGKAH SELANJUTNYA:')
print('  1. Download model.pkl dan scaler.pkl dari file explorer Colab')
print('  2. Upload ke repository GitHub kamu')
print('  3. Deploy ke Streamlit Community Cloud')

# Download otomatis ke komputer
files.download('model.pkl')
files.download('scaler.pkl')

## 🧪 8. Uji Prediksi Manual

In [ ]:
# Contoh prediksi satu siswa
LABEL_NAMES = {0: '❌ Tidak Siap', 1: '⚡ Potensial', 2: '🏆 Siap Olimpiade'}

def prediksi_siswa(aljabar, geometri, bilangan, data_kt, menalar, literasi):
    X_new = np.array([[aljabar, geometri, bilangan, data_kt, menalar, literasi]])
    X_new_scaled = scaler.transform(X_new)
    pred = model.predict(X_new_scaled)[0]
    proba = model.predict_proba(X_new_scaled)[0]
    avg = np.mean([aljabar, geometri, bilangan, data_kt, menalar, literasi])

    print('=' * 45)
    print(f'  STATUS  : {LABEL_NAMES[pred]}')
    print(f'  Rata-rata Skor   : {avg:.1f}')
    print(f'  Tingkat Keyakinan: {proba[pred]*100:.1f}%')
    print('  Probabilitas per kelas:')
    for i, name in LABEL_NAMES.items():
        print(f'    {name}: {proba[i]*100:.1f}%')
    print('=' * 45)

# Contoh penggunaan
print('\n📌 Contoh 1 — Siswa berprestasi tinggi:')
prediksi_siswa(aljabar=88, geometri=85, bilangan=90, data_kt=82, menalar=87, literasi=91)

print('\n📌 Contoh 2 — Siswa rata-rata:')
prediksi_siswa(aljabar=65, geometri=62, bilangan=68, data_kt=60, menalar=63, literasi=70)

print('\n📌 Contoh 3 — Siswa perlu pembinaan:')
prediksi_siswa(aljabar=42, geometri=38, bilangan=45, data_kt=40, menalar=35, literasi=50)

---
## ✅ Selesai!

File yang sudah di-download:
- `model.pkl` — Model Random Forest terlatih
- `scaler.pkl` — StandardScaler untuk normalisasi

Selanjutnya upload kedua file ini ke GitHub bersama `app.py` dan `requirements.txt`.